# MRI val/test recompute + ViT-Base T1d on Colab A100

Fills the cross-model **val AUC/F1** gaps without the HPC queue. Two jobs:

1. **Recompute** val+test from the saved `best_model.pt` for BrainMVP / AG-MS3D / 3D-CNN (`09_collect_val_test_metrics.py --val_test`). Preserves the published test numbers.
2. **Train ViT-Base T1d** (seeds 0/1/2) — the unfinished cell currently shown as a 0.500 placeholder.

**Stage to Drive first** (from your local machine): the run trees with `best_model.pt` + `metrics.json` (from `D:`), the input volumes (`brainmvp_inputs`, `cnn_inputs` from HPC; `vit_inputs` from `D:`), the splits dir, the matched-labels CSV, and `BrainMVP_uniformer.pt` / the ViT MAE pretrained ckpt.

After it runs, sync the patched `metrics.json` (+ ViT-Base run dirs) back to `D:` and re-run `06` → `06c` locally.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi -L  # confirm A100

In [ ]:
# Repo (trainer code). Pull latest main.
%cd /content
![ -d Transformers_XAI ] && (cd Transformers_XAI && git pull) || git clone https://github.com/econci474/Transformers_XAI.git
%cd /content/Transformers_XAI
# Heavy deps: install once to Drive (per project convention) or pip here.
!pip -q install monai nibabel einops timm 2>/dev/null; echo done

In [ ]:
# >>> EDIT these to your Drive staging paths <<<
STAGE        = '/content/drive/MyDrive/ADNI'
DERIVS_ROOT  = f'{STAGE}/derivatives'          # holds brainmvp_debug/, cnn3d_outputs/, agms3d_outputs/, vit_baseline/
BRAINMVP_INPUTS = f'{STAGE}/brainmvp_inputs'
CNN_INPUTS      = f'{STAGE}/cnn_inputs'
VIT_INPUTS      = f'{STAGE}/vit_inputs'
DATA_DIR        = f'{STAGE}/no_cdr_stratified_post_exclusion/tabular/baseline'
MATCHED_LABELS  = f'{STAGE}/master_mri_clinical_matched_viscode2_extended_post_exclusion.csv'
BRAINMVP_CKPT   = f'{STAGE}/ViT_pretrained/BrainMVP_uniformer.pt'
VIT_MAE_CKPT    = f'{STAGE}/ViT_pretrained/ViT_B_pretrained_noaug_mae75_BRATS2023_IXI_OASIS3_seed_8456_999_077000.pth.tar'
import os; [print('OK ' if os.path.exists(p) else 'MISSING ', p) for p in
  [DERIVS_ROOT, DATA_DIR, MATCHED_LABELS, BRAINMVP_CKPT]]

## 1. Recompute val+test (dry-run first, then real)

In [ ]:
!python mri_pipeline/09_collect_val_test_metrics.py \
  --derivs-root '{DERIVS_ROOT}' --brainmvp-inputs '{BRAINMVP_INPUTS}' \
  --cnn-inputs '{CNN_INPUTS}' --data-dir '{DATA_DIR}' \
  --matched-labels '{MATCHED_LABELS}' --brainmvp-ckpt '{BRAINMVP_CKPT}' \
  --models brainmvp cnn3d agms3d --only-missing --dry-run

In [ ]:
# Real run (drop --dry-run). Patches a val_metrics block into each metrics.json.
!python mri_pipeline/09_collect_val_test_metrics.py \
  --derivs-root '{DERIVS_ROOT}' --brainmvp-inputs '{BRAINMVP_INPUTS}' \
  --cnn-inputs '{CNN_INPUTS}' --data-dir '{DATA_DIR}' \
  --matched-labels '{MATCHED_LABELS}' --brainmvp-ckpt '{BRAINMVP_CKPT}' \
  --models brainmvp cnn3d agms3d --only-missing

## 2. Train ViT-Base T1d (seeds 0/1/2)
Same recipe as the existing ViT-Base (scratch) T1/T1b/T2 rows; the ViT trainer logs val auc+f1.
Outputs land under `vit_baseline/ViT_B_scratch/T1d_binary/seed_*/baseline/`.

In [ ]:
VIT_BASE_OUT = f'{DERIVS_ROOT}/vit_baseline'
for seed in (0, 1, 2):
    !python mri_pipeline/04_supervised_finetuning_ViT.py \
      --task T1d_binary --seed {seed} --strategy scratch --vit_size base \
      --augment random --vit_inputs_dir '{VIT_INPUTS}' \
      --data_dir '{DATA_DIR}' --matched_labels_csv '{MATCHED_LABELS}' \
      --out_dir '{VIT_BASE_OUT}' --num_workers 2

## 3. Sync back
From your **local** machine, pull the patched metrics + ViT-Base runs from Drive back onto `D:` (rsync/Drive client), then locally:
```
python mri_pipeline/06_render_cross_model_table.py
python mri_pipeline/06c_render_styled_cross_model.py
```
and the val AUC/F1 cells (and the ViT-Base T1d row) fill in.